# Routing: Classify and Dispatch to Specialists

**What you'll learn:**
- How routing enables separation of concerns in LLM systems
- Implementing classification-based dispatch with chain-of-thought
- When routing outperforms a single generalist prompt
- A cost optimization pattern: routing by model complexity

**Position on the spectrum:** Routing adds branching to your workflows — the system picks ONE specialized path per input.

> *"Routing classifies an input and directs it to a specialized follow-up task. This enables separation of concerns — each downstream path can have an optimized prompt, model, or toolset."*

## How It Works

![Routing workflow — LLM Call Router dispatches to one of several specialized LLM paths](assets/routing.webp)

**The flow:** Input → Router (classify) → Specialized Handler A, B, or C → Output

**Key concepts:**
- A **router** LLM (or classifier) categorizes the input
- Each downstream path has its own optimized prompt, model, or tools
- Only **one path** executes per input (unlike parallelization)
- Classification accuracy is the ceiling on routing quality

**Why it works better than a single prompt:**
- Specialized prompts can be longer and more detailed without confusing other categories
- Different categories may need different tools, context, or even different models
- You can optimize and test each route independently

## When to Use Routing

✅ **Use when:**
- There are distinct categories of input that benefit from different handling
- Classification can be handled accurately (by LLM or traditional classifier)  
- Optimizing prompts for one category would hurt performance on another
- Different inputs need different tools or context

❌ **Don't use when:**
- Inputs don't fall into discrete categories (use [Prompt Chaining](01_prompt_chaining.ipynb))
- All inputs need the same handling regardless of content
- Classification is unreliable for your input distribution

In [ ]:
import sys
sys.path.append(".")
from util import llm_call, extract_xml

In [ ]:
def route(input_text: str, routes: dict[str, str]) -> str:
    """Route input to a specialized handler based on content classification.
    
    Args:
        input_text: The input to classify and process
        routes: Dict mapping route names to specialized system prompts
    
    Returns:
        Response from the selected specialized handler
    """
    print(f"Available routes: {list(routes.keys())}")
    
    # Step 1: Classify with chain-of-thought reasoning
    selector_prompt = f"""Analyze the input and select the most appropriate handler from: {list(routes.keys())}

First explain your reasoning, then provide your selection in XML format:

<reasoning>
Why this input should be routed to a specific handler. Consider key terms, intent, and urgency.
</reasoning>

<selection>
The chosen handler name (must exactly match one of the available options)
</selection>

Input: {input_text}"""

    route_response = llm_call(selector_prompt)
    reasoning = extract_xml(route_response, "reasoning")
    route_key = extract_xml(route_response, "selection").strip().lower()
    
    print(f"\nRouting reasoning: {reasoning.strip()}")
    print(f"\n→ Selected route: {route_key}")
    
    # Step 2: Process with specialized handler
    if route_key not in routes:
        print(f"  ⚠ Unknown route '{route_key}', falling back to first available")
        route_key = list(routes.keys())[0]
    
    selected_prompt = routes[route_key]
    result = llm_call(f"{selected_prompt}\nInput: {input_text}")
    return result

## Example 1: Customer Support Router

The classic routing use case — different support categories need different expertise, tone, and procedures. A billing question needs financial context; a technical issue needs troubleshooting steps.

In [ ]:
support_routes = {
    "billing": """You are a billing support specialist. Follow these guidelines:
    1. Acknowledge the specific billing concern
    2. Explain charges or discrepancies clearly
    3. List concrete next steps with timeline
    4. Offer payment options if relevant
    Keep responses professional and concise.""",
    
    "technical": """You are a technical support engineer. Follow these guidelines:
    1. List exact steps to resolve the issue
    2. Include system requirements if relevant
    3. Provide workarounds for common problems
    4. End with escalation path if needed
    Use numbered steps and technical details.""",
    
    "account": """You are an account security specialist. Follow these guidelines:
    1. Prioritize account security and verification
    2. Provide clear steps for account recovery
    3. Include security tips and warnings
    4. Set clear expectations for resolution time
    Maintain a security-focused tone.""",
    
    "product": """You are a product specialist. Follow these guidelines:
    1. Focus on feature education and best practices
    2. Include specific examples of usage
    3. Suggest related features that might help
    4. Be educational and encouraging in tone.""",
}

# Test with a billing ticket
ticket = """Subject: Unexpected charge on my card
Message: I just noticed a charge of $49.99 but I thought I was on the $29.99 plan.
Can you explain this and adjust if it's a mistake? Thanks, Sarah"""

print("=" * 60)
print("TICKET:")
print(ticket)
print("=" * 60)
response = route(ticket, support_routes)
print(f"\n{'─' * 60}")
print("RESPONSE:")
print(f"{'─' * 60}")
print(response)

## Example 2: Model Selection Router (Cost Optimization)

A powerful routing application: route easy queries to cheaper, faster models and complex queries to more capable (expensive) ones. This can cut API costs by 50-80% while maintaining quality where it matters.

This is explicitly recommended in Anthropic's guidance:
> *"Route easy/common questions to smaller cost-efficient models (e.g., Claude Haiku) and hard/unusual questions to more capable models (e.g., Claude Sonnet)"*

In [ ]:
def model_router(query: str) -> str:
    """Route queries to appropriate models based on complexity."""
    
    # Classification prompt
    classifier_prompt = f"""Classify this query's complexity level for an AI assistant:

SIMPLE: Factual lookups, basic formatting, simple math, greetings, short translations
COMPLEX: Multi-step reasoning, code generation, analysis, creative writing, nuanced questions

<reasoning>Brief analysis of query complexity</reasoning>
<complexity>SIMPLE or COMPLEX</complexity>

Query: {query}"""

    classification = llm_call(classifier_prompt)
    complexity = extract_xml(classification, "complexity").strip().upper()
    reasoning = extract_xml(classification, "reasoning").strip()
    
    # Route to appropriate model
    if complexity == "SIMPLE":
        model = "claude-haiku-4-5-20251001"
        cost_note = "~$0.25/M input tokens"
    else:
        model = "claude-sonnet-4-6"
        cost_note = "~$3/M input tokens"
    
    print(f"Query: {query}")
    print(f"Analysis: {reasoning}")
    print(f"Complexity: {complexity} → Model: {model} ({cost_note})")
    print(f"{'─' * 50}")
    
    response = llm_call(query, model=model)
    return response


# Test with queries of varying complexity
queries = [
    "What is the capital of France?",
    "Explain the trade-offs between microservices and monolithic architectures for a startup with 5 engineers.",
]

for q in queries:
    print(f"\n{'═' * 60}")
    result = model_router(q)
    print(f"\nResponse preview: {result[:200]}...")

## Pitfalls & Design Considerations

| Pitfall | Impact | Mitigation |
|---------|--------|------------|
| **Classification errors cascade** | Misrouted input gets inappropriate handling with no recovery | Add confidence thresholds + fallback route |
| **Too many routes** | Classification difficulty increases, accuracy drops | Keep routes ≤ 5-7; merge similar categories |
| **Overlapping categories** | Router can't reliably distinguish between routes | Make categories mutually exclusive with clear boundaries |
| **No fallback path** | Unclassifiable inputs get force-fitted into a wrong route | Add an "other/general" catch-all route |

### ACI Principle Applied: Clear Category Definitions

The quality of your route descriptions is a direct application of the Agent-Computer Interface principle. Compare:

**Bad:** `"billing"` — Vague, model guesses what belongs here  
**Good:** `"billing — Questions about charges, invoices, payment methods, subscription plans, refunds, or pricing discrepancies"` — Clear boundaries

## Routing vs. Other Patterns

| Scenario | Pattern | Why |
|----------|---------|-----|
| Input has ONE correct handler | **Routing** | Classify once, process once |
| Input needs ALL handlers | **Parallelization** | Every perspective matters |
| Input needs handlers in sequence | **Prompt Chaining** | Order matters |
| Don't know which handlers until runtime | **Orchestrator-Workers** | Dynamic decomposition |

## Key Takeaways

1. **Routing = classification + specialized dispatch** — one path per input
2. **Classification accuracy is your ceiling** — invest in clear category definitions
3. **Separation of concerns pays off** — each route can be optimized independently
4. **Model routing saves money** — route simple queries to cheaper models

---

**Next up:** [03_parallelization.ipynb](03_parallelization.ipynb) — Run multiple LLM calls simultaneously for speed and confidence.